# PHOENIX14T–CSLR: 68 lượt chạy — Colab

Run All chuẩn bị dữ liệu chung rồi chạy 67 cấu hình định lượng và một baseline cho so sánh gloss định tính. Mỗi thí nghiệm có checkpoint riêng trên Drive. Chạy lại tiếp tục thí nghiệm dở, bỏ qua thí nghiệm hoàn tất. Đây là tái dựng theo paper, có giả định công khai trong `docs/PHOENIX14T_SUITE.md`; chưa chứng minh đạt số paper.

## Chuẩn bị Drive một lần
Đưa `SignLanguage-Reproduce-portable.zip` vào `MyDrive/SignLanguage-Reproduction/`. Notebook tự tải và lưu archive PHOENIX14T vào cùng thư mục Drive nếu chưa có; bạn không cần tự tải hoặc giải nén dữ liệu. Cần Internet và khoảng 39 GB trống trên Drive cho archive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cấu hình

Mặc định chạy đủ 68 lượt PHOENIX14T–CSLR. Giữ EPOCHS=50; SESSION_MINUTES giới hạn mỗi phiên. Nếu đổi epoch, đổi CAMPAIGN_NAME để giữ tách biệt với chiến dịch 50 epoch.

In [ ]:
from pathlib import Path
import time
SHARED_DRIVE_ROOT=Path("/content/drive/MyDrive/SignLanguage-Reproduction")
PROJECT_ZIP=SHARED_DRIVE_ROOT/"SignLanguage-Reproduce-portable.zip"
WORKSPACE=Path("/content")
DATA_ROOT=Path("/content/phoenix14t")
PHOENIX_ARCHIVE=SHARED_DRIVE_ROOT/"phoenix-2014-T.v3.tar.gz"
MANIFEST_ROOT=SHARED_DRIVE_ROOT/"manifests/phoenix14t"

DATASET="phoenix14t"
PHASE="cslr"
WORKERS=2
EPOCHS=50
SESSION_MINUTES=600  # entire notebook session, including preparation
ONLY_EXPERIMENTS=""  # empty: all 68; e.g. "main,modules_none"
CAMPAIGN_NAME="phoenix14t_cslr_suite_cpu_storage_v11_seed0"
NOTEBOOK_START=time.monotonic()

## Cài mã và thư viện

Notebook chỉ cài thư viện vào runtime tạm của Colab. Nó không sửa máy tính cá nhân.

In [ ]:
from pathlib import Path
import hashlib, zipfile, shutil, subprocess, sys

if not PROJECT_ZIP.is_file():
    raise FileNotFoundError(f"Upload the project ZIP here first: {PROJECT_ZIP}")
project_root = WORKSPACE / "SignLanguage-Reproduce"
if True:  # refresh source from the current Drive ZIP
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PROJECT_ZIP) as archive:
        for entry in archive.infolist():
            target=(WORKSPACE / entry.filename).resolve()
            if not target.is_relative_to(WORKSPACE.resolve()):
                raise ValueError(f"Unsafe ZIP path: {entry.filename}")
        archive.extractall(WORKSPACE)
if not (project_root / "scripts/launch.py").is_file():
    raise FileNotFoundError("Project ZIP has an unexpected layout")
subprocess.run([sys.executable, "scripts/verify_bundle.py"],cwd=project_root,check=True)
print("Project checksum verified:", project_root)

In [ ]:
%pip install -q -r /content/SignLanguage-Reproduce/requirements-notebook-cslr.txt

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/fetch_phoenix.py', '--data-root', str(DATA_ROOT), '--archive', str(PHOENIX_ARCHIVE)], cwd=project_root, check=True)


In [ ]:
from pathlib import Path
import subprocess, sys

if DATASET in ("phoenix14t","phoenix14") and not (project_root/"assets/resnet18-f37072fd.pth").is_file():
    subprocess.run([sys.executable,"scripts/fetch_assets.py","--resnet","--output",
                    str(project_root/"assets")],cwd=project_root,check=True)
if PHASE in ("slt","gfslt") and not (project_root/"assets/mbart-large-cc25/config.json").is_file():
    subprocess.run([sys.executable,"scripts/fetch_assets.py","--mbart",
        "--mbart-revision","f417e5563320b2cc8aabe4329d986b238809067f",
        "--output",str(project_root/"assets")],cwd=project_root,check=True)

## Dữ liệu
Notebook tự tải archive PHOENIX14T từ máy chủ RWTH vào Drive nếu chưa có, rồi giải nén vào ổ tạm Colab để tránh tạo hàng trăm nghìn file nhỏ trực tiếp trên Drive. Archive trên Drive giúp lần chạy sau không phải tải lại từ nguồn. Cần khoảng 39 GB trống trên Drive và đủ dung lượng runtime để giải nén.

In [ ]:
import subprocess, sys
from pathlib import Path
manifest_dir=Path(MANIFEST_ROOT)
subprocess.run([sys.executable,"scripts/prepare_data.py","--dataset","phoenix14t",
    "--annotations",str(DATA_ROOT/"annotations/manual"),"--data-root",str(DATA_ROOT),
    "--output",str(manifest_dir),"--overwrite"],cwd=project_root,check=True)
for kind in ("swin_t","pvig_tiny"):
    subprocess.run([sys.executable,"scripts/fetch_backbones.py","--backbone",kind],cwd=project_root,check=True)
import torch, torchvision
if not torch.cuda.is_available(): raise RuntimeError("Enable GPU before running the experiment suite")
print("GPU:",[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("torch:",torch.__version__,"torchvision:",torchvision.__version__)

## Hàng đợi thí nghiệm

Kết quả ở Drive `runs/phoenix14t_cslr_suite_cpu_storage_v11_seed0/`, mỗi cấu hình một thư mục. Checkpoint lưu sau mỗi 200 optimizer updates và khi kết thúc epoch/dừng phiên có kiểm soát. Phiên bị nền tảng ngắt đột ngột sẽ tiếp tục từ checkpoint đã commit gần nhất.

In [ ]:
import os, subprocess, sys, tempfile, time
from pathlib import Path
env=os.environ.copy()
env["HF_HUB_OFFLINE"]="1"; env["TRANSFORMERS_OFFLINE"]="1"; env["PYTHONUNBUFFERED"]="1"
suite_output=project_root/"runs"/CAMPAIGN_NAME
args=[sys.executable,"scripts/run_suite.py","--data-root",str(DATA_ROOT),
      "--manifests",str(MANIFEST_ROOT),"--output",str(suite_output),
      "--minutes",str(max(3,SESSION_MINUTES-(time.monotonic()-NOTEBOOK_START)/60)),"--epochs",str(EPOCHS),"--workers",str(WORKERS)]
if ONLY_EXPERIMENTS: args += ["--only",ONLY_EXPERIMENTS]
credentials_file=None
try:
    if "drive_service" in globals():
        from kaggle_secrets import UserSecretsClient
        fd,name=tempfile.mkstemp(prefix="signlanguage-private-",suffix=".json",dir="/tmp")
        credentials_file=Path(name)
        with os.fdopen(fd,"w") as f:
            f.write(UserSecretsClient().get_secret("GOOGLE_DRIVE_OAUTH_JSON"))
        os.chmod(credentials_file,0o600)
        env["SIGN_DRIVE_CREDENTIALS"]=str(credentials_file)
        args += ["--drive-folder",DRIVE_WORKSPACE_FOLDER_ID]
    else:
        args += ["--persist-root",str(SHARED_DRIVE_ROOT/"runs"/CAMPAIGN_NAME)]
    process=subprocess.Popen(args,cwd=project_root,env=env)
    try:
        code=process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try: process.wait(timeout=120)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    if code: raise RuntimeError("Campaign stopped with an error; inspect output. Saved checkpoints remain available.")
    print("Session finished. Run All again to resume pending experiments; see suite_summary.csv on Drive.")
finally:
    if credentials_file: credentials_file.unlink(missing_ok=True)

## Kết quả

Đọc `suite_summary.csv`: pending là chưa hoàn tất, measured reconstruction là số đo thật. Không có số đo thì không tự điền số paper. Phiên mới có thể phải giải nén lại archive đã cache trên Drive; không cần tải lại từ RWTH.

## Bản cập nhật không CI
Notebook vẫn dùng Run all và thư mục Drive chung; không cần thêm secret W&B. Khi code đổi, thay ZIP mới trong Drive trước phiên chạy mới. Không chạy Colab và Kaggle đồng thời vào cùng chiến dịch.
V11 chuyển activation CUDA train sang RAM để giảm VRAM; không đổi nhãn, frame hoặc batch. Tốn thêm RAM và có thể chậm hơn. Tên CAMPAIGN_NAME mặc định mới giữ kết quả/checkpoint các bản cũ; không dùng tên chiến dịch cũ với fingerprint code mới.
Máy Linux của cô có luồng riêng: chạy `setup_machine.py` một lần, sau đó `run.py` để tự lấy bản sửa từ GitHub và gửi lỗi lên W&B. Xem `docs/MAY_CO_VA_LAPTOP.md`. Cả ba nơi dùng cùng catalog PHOENIX14T–CSLR; chưa có số đo thật để khẳng định tái lập paper.